# DecodingExample

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `smoke`
- Workflow family: `decoding_1d`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/DecodingExample.md`


Notebook source link: [DecodingExample.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/DecodingExample.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "DecodingExample"
FAMILY = "decoding_1d"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"DecodingExample: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"DecodingExample: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"DecodingExample: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"DecodingExample: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "delta = 0.001; Tmax = 10;",
    "time = 0:delta:Tmax;",
    "f=.1; b1=1;b0=-3;",
    "x = sin(2*pi*f*time);",
    "expData = exp(b1*x+b0);",
    "lambdaData = expData./(1+expData);",
    "lambda = Covariate(time,lambdaData./delta, '\\Lambda(t)','time','s','Hz',{'\\lambda_{1}'},{{' ''b'', ''LineWidth'' ,2'}});",
    "numRealizations = 10;",
    "spikeColl = CIF.simulateCIFByThinningFromLambda(lambda,numRealizations);",
    "figure;",
    "subplot(2,1,1); spikeColl.plot;",
    "subplot(2,1,2); lambda.plot;",
    "stim = Covariate(time,sin(2*pi*f*time),'Stimulus','time','s','V',{'stim'});",
    "baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',...",
    "{'constant'});",
    "figure;",
    "cc = CovColl({stim,baseline});",
    "trial = Trial(spikeColl,cc);",
    "trial.plot;",
    "clear c;",
    "selfHist = [] ; NeighborHist = []; sampleRate = 1000;",
    "c{1} = TrialConfig({{'Baseline','constant'}},sampleRate,selfHist,...",
    "NeighborHist);",
    "c{1}.setName('Baseline');",
    "c{2} = TrialConfig({{'Baseline','constant'},{'Stimulus','stim'}},...",
    "sampleRate,selfHist,NeighborHist);",
    "c{2}.setName('Baseline+Stimulus');",
    "cfgColl= ConfigColl(c);",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);",
    "figure;",
    "results{1}.plotResults;",
    "Summary = FitResSummary(results);",
    "paramEst = squeeze(Summary.bAct(:,2,:));",
    "meanParams = mean(paramEst,2);",
    "clear lambdaCIF;",
    "b0=paramEst(1,:);",
    "b1=paramEst(2,:);",
    "for i=1:numRealizations",
    "lambdaCIF{i} = CIF([b0(i) b1(i)],{'1','x'},{'x'},'binomial');",
    "end",
    "spikeColl.resample(1/delta);",
    "dN=spikeColl.dataToMatrix;",
    "Q=2*std(stim.data(2:end)-stim.data(1:end-1));",
    "A=1;",
    "[x_p, W_p, x_u, W_u] = DecodingAlgorithms.PPDecodeFilterLinear(A, Q, dN',b0,b1,'binomial',delta);",
    "figure;",
    "zVal=3;",
    "ciLower = min(x_u(1:end)-zVal*squeeze(sqrt(W_u(1:end)))',x_u(1:end)+zVal*squeeze(sqrt(W_u(1:end)))');",
    "ciUpper = max(x_u(1:end)-zVal*squeeze(sqrt(W_u(1:end)))',x_u(1:end)+zVal*squeeze(sqrt(W_u(1:end)))');",
    "hEst=plot(time,x_u(1:end),'b',time,ciLower,'g',time,ciUpper,'g'); hold on;",
    "hold all;",
    "hStim=stim.plot([],{{' ''k'',''Linewidth'',2'}});",
    "legend off;",
    "legend([hEst(1) hEst(2) hEst(3) hStim],'x_{k|k}(t)',strcat('x_{k|k}(t)-',num2str(zVal),'\\sigma_{k|k}'),...",
    "strcat('x_{k|k}(t)+',num2str(zVal),'\\sigma_{k|k}'),'x(t)');",
    "title(['Decoded Stimulus +/- 99% confidence intervals using ' num2str(numRealizations) ' cells']);"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for DecodingExample.")


In [ ]:
# 1D Decoding workflow: decode latent state sequence from population spikes.
n_units = 14
n_states = 17
n_time = 260
state_idx = np.arange(n_states)

transition = np.zeros((n_states, n_states), dtype=float)
for i in range(n_states):
    for j, w in ((i - 1, 0.2), (i, 0.6), (i + 1, 0.2)):
        if 0 <= j < n_states:
            transition[i, j] += w
    transition[i, :] /= np.sum(transition[i, :])

latent = np.zeros(n_time, dtype=int)
latent[0] = n_states // 2
for t in range(1, n_time):
    latent[t] = rng.choice(n_states, p=transition[latent[t - 1]])

centers = np.linspace(0.0, n_states - 1, n_units)
widths = np.full(n_units, 2.1)
state_axis = np.arange(n_states)[None, :]
tuning = 0.06 + 0.42 * np.exp(-0.5 * ((state_axis - centers[:, None]) / widths[:, None]) ** 2)

use_history = TOPIC in {"DecodingExampleWithHist", "nSTATPaperExamples"}

if use_history:
    gain = np.ones(n_time, dtype=float)
    counts = np.zeros((n_units, n_time), dtype=float)
    prev = 0.0
    for t in range(n_time):
        gain[t] = np.exp(0.50 * prev)
        lam = tuning[:, latent[t]] * gain[t]
        counts[:, t] = rng.poisson(lam)
        prev = float(np.mean(counts[:, t]))

    decoded_raw, _ = DecodingAlgorithms.decode_state_posterior(counts, tuning, transition)
    corrected = counts / gain[None, :]
    decoded, posterior = DecodingAlgorithms.decode_state_posterior(corrected, tuning, transition)
    rmse_raw = float(np.sqrt(np.mean((decoded_raw - latent) ** 2)) / (n_states - 1))
    rmse_dec = float(np.sqrt(np.mean((decoded - latent) ** 2)) / (n_states - 1))
else:
    counts = np.zeros((n_units, n_time), dtype=float)
    for t in range(n_time):
        counts[:, t] = rng.poisson(tuning[:, latent[t]])
    decoded, posterior = DecodingAlgorithms.decode_state_posterior(counts, tuning, transition)
    rmse_raw = float(np.sqrt(np.mean((decoded - latent) ** 2)) / (n_states - 1))
    rmse_dec = rmse_raw

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(latent, label="true", linewidth=1.2)
axes[0].plot(decoded, label="decoded", linewidth=1.0)
axes[0].set_title(f"{TOPIC}: latent-state decoding")
axes[0].set_ylabel("state")
axes[0].legend(loc="upper right")

im = axes[1].imshow(posterior, aspect="auto", origin="lower", cmap="viridis")
axes[1].set_title("Posterior over latent states")
axes[1].set_xlabel("time bin")
axes[1].set_ylabel("state")
fig.colorbar(im, ax=axes[1], fraction=0.03, pad=0.02)

plt.tight_layout()
plt.show()

print("rmse_raw", rmse_raw, "rmse_final", rmse_dec)
assert np.max(np.abs(np.sum(posterior, axis=0) - 1.0)) < 1e-6
if use_history:
    assert rmse_dec <= rmse_raw + 0.03

CHECKPOINT_METRICS = {
    "rmse_raw": float(rmse_raw),
    "rmse_dec": float(rmse_dec),
}
CHECKPOINT_LIMITS = {
    "rmse_raw": (0.0, 0.65),
    "rmse_dec": (0.0, 0.65),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
